# Notebook 04 — Machine Learning for Credit Risk

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Credit scoring is the oldest and most successful application of statistical
learning in finance. It also has a property that market prediction does not:
**the signal is real**. Applicants with no checking account and a history of
missed payments really do default more often. That makes it the right place to
learn the machinery, because when something does not work you can be
reasonably confident the problem is your code rather than the universe.

**The data.** The Statlog (German Credit) dataset — 1,000 loan applications
from a German bank, 20 attributes each, with a known good/bad outcome. It has
been a teaching and benchmarking standard since 1994.

**What you will learn**

1. Exploring a categorical-heavy dataset
2. Fair lending: which variables you are legally forbidden to use
3. `ColumnTransformer` and `Pipeline` — encoding without leakage
4. Logistic regression, and reading coefficients as odds ratios
5. Tree-based models: decision tree, random forest, gradient boosting
6. Why accuracy is the wrong metric, and what to use instead
7. Cross-validation
8. Choosing a cut-off from expected profit, not from 0.5

**How to use this notebook.** Every line of code is already written and commented. Run each cell in
order (Shift+Enter), read the explanation above it, and check that the output matches what the text
says to look for. The four exercises come with worked solutions. When you have finished, take the
Canvas comprehension quiz for this notebook.

## 0. Setup

The next cell loads the libraries and finds the data folder. You met all of these in Notebooks 01–03,
so only a quick recap:

- **NumPy** (`np`) — fast numerical arrays. Here it supplies `np.exp` (odds ratios), `np.where`
  (profit per loan) and `np.linspace` (a grid of cut-offs).
- **pandas** (`pd`) — labelled tables. Every dataset in this notebook is a pandas `DataFrame`.
- **matplotlib** (`plt`) — the plotting library underneath almost every Python chart.
- **pathlib** — file paths as objects, so the code runs unchanged on Windows and macOS.

`RANDOM_STATE = 42` matters more here than in earlier notebooks. Splitting the data, building a
random forest and shuffling cross-validation folds all use random numbers. Fixing the seed makes them
repeatable, so your numbers match the text and your classmates'. Nothing produces output; a silent
cell means success.

In [ ]:
# pathlib.Path represents file paths as objects, so "/" joins folders the same way on Windows and Mac.
from pathlib import Path

# NumPy: fast arrays and maths (np.exp, np.where, np.linspace are used below).
import numpy as np
# pandas: tables (DataFrames) with labelled rows and columns — the analyst's spreadsheet in code.
import pandas as pd
# matplotlib.pyplot: the standard Python plotting library; every chart in this notebook uses it.
import matplotlib.pyplot as plt


# A helper that finds the project folder, so the notebook works wherever it was opened from.
def find_repo_root(start=None):
    # Docstring: one-line description of what the function returns.
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    # Start from the given folder (or the current working folder) and make the path absolute.
    here = Path(start or Path.cwd()).resolve()
    # Walk upwards: the folder itself, then its parent, grandparent and so on.
    for candidate in [here, *here.parents]:
        # The repository root is the first folder that contains data/sp500_prices.csv.
        if (candidate / "data" / "sp500_prices.csv").exists():
            # Found it: hand the folder back to the caller.
            return candidate
    # Reached the top of the drive without finding it: stop with a clear, actionable message.
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "  # Python joins adjacent strings into one message
        "and open the mtu4600-lab3-analytics folder itself, then re-run."  # second half of the message
    )


# Locate the repository root once and keep it in REPO.
REPO = find_repo_root()
# The data folder, built with the "/" path operator.
DATA = REPO / "data"
# Apply the course's shared chart style (fonts, colours, grid) to every plot below.
plt.style.use(REPO / "fin4600.mplstyle")

# Show up to 40 columns when a DataFrame is displayed, instead of hiding the middle ones with "...".
pd.set_option("display.max_columns", 40)
# Allow 150 characters per printed line before pandas wraps a wide table.
pd.set_option("display.width", 150)

# Fix the seed for every random step (the split, the forest, CV folds) so everyone gets the same numbers.
RANDOM_STATE = 42  # fix the seed so everyone in the class gets the same numbers

Load the German Credit data with `pd.read_csv` and take a first look.

The target column is `default`: **1** if the loan went bad, **0** if it was repaid. Because it is
coded 0/1, its **mean is the default rate**, a trick used throughout this notebook. Look for
`Shape: (1000, 22)` and a default rate of **30.0%**. That 30% is the *base rate*, and you will keep
comparing models with it.

In [ ]:
# Read the German Credit file into a DataFrame: one row per loan application, one column per attribute.
credit = pd.read_csv(DATA / "german_credit.csv")

# .shape is a (rows, columns) tuple — expect (1000, 22).
print("Shape:", credit.shape)
# "default" is 1 for a bad loan and 0 for a good one, so its mean is the share that defaulted (":.1%" formats as a percentage).
print(f"Default rate: {credit['default'].mean():.1%}")
# Show the first five rows; as the last line of the cell, Jupyter displays it as a table.
credit.head()

## 1. What is in the file?

`DataFrame.info()` lists each column's name, how many non-missing values it has and its data type.
Look for two things: there are **no missing values** (every column shows 1000 non-null), and most
columns are **text** (`str`). Text columns cannot go straight into a model; section 5 deals with that.

In [ ]:
# List every column with its non-null count and data type (int64 = whole numbers, str/object = text).
credit.info()

Sort the columns into **numeric** and **categorical** lists, because each kind needs different
preprocessing later.

`select_dtypes(include="number")` keeps the numeric columns and `exclude="number"` keeps the rest.
`.columns.drop(...)` removes a name from the list of columns. We remove `default`, because it is the
target and must never be a feature, and `applicant_id`, because an ID number is arbitrary. A model
fed an ID would just memorise it. Expect 7 numeric and 13 categorical features.

In [ ]:
# select_dtypes(include="number") keeps the numeric columns; .columns gives their names; drop the target; .tolist() makes a plain list.
numeric_columns = credit.select_dtypes(include="number").columns.drop("default").tolist()
# exclude="number" keeps the text columns; drop applicant_id because an ID number carries no credit information.
categorical_columns = credit.select_dtypes(exclude="number").columns.drop("applicant_id").tolist()

# len() counts the list; the f-string drops the count into the sentence.
print(f"{len(numeric_columns)} numeric features:")
# ", ".join(...) glues the names into one comma-separated line.
print("   ", ", ".join(numeric_columns))
# "\n" starts with a blank line to separate the two lists.
print(f"\n{len(categorical_columns)} categorical features:")
# Same for the text (categorical) columns.
print("   ", ", ".join(categorical_columns))

`describe()` summarises each numeric column: count, mean, standard deviation, minimum, quartiles
and maximum. Note the different **scales**. `amount` runs from 250 to 18,424 DM while
`installment_rate` only takes the values 1–4. A model that measures distances or penalises
coefficients would be dominated by `amount` unless we rescale, which is why `StandardScaler` appears
in section 5.

In [ ]:
# describe() gives count, mean, std, min, quartiles and max for each numeric column; round(1) keeps it readable.
credit[numeric_columns].describe().round(1)

## 2. Which variables look like they matter?

Before fitting anything, look at the default rate within each category. This
is the credit analyst's version of exploratory data analysis, and it is often
where most of the insight comes from.

The helper `default_rate_by` uses the `groupby` + `agg` pattern from Notebook 01. It splits the rows
by category, then computes two named outputs per group: `applications` (the group's size) and
`default_rate` (the mean of the 0/1 target). Look at the gap between the safest and riskiest
**checking-account** categories.

In [ ]:
# Reusable helper: for any column, count applications and compute the default rate in each category.
def default_rate_by(column: str) -> pd.DataFrame:
    # groupby splits the rows by category; agg with named outputs builds one row per category.
    table = credit.groupby(column).agg(
        applications=("default", "size"),  # number of rows in the group
        default_rate=("default", "mean"),  # mean of a 0/1 column = share that defaulted
    )
    # Turn the 0-1 fraction into a percentage with one decimal place.
    table["default_rate"] = (table["default_rate"] * 100).round(1)
    # Riskiest category first.
    return table.sort_values("default_rate", ascending=False)


# to_string() prints the whole table without truncation; "\n" adds line breaks around it.
print("Checking account status\n", default_rate_by("status").to_string(), "\n")
# The same breakdown for credit history.
print("Credit history\n", default_rate_by("credit_history").to_string())

Read those tables carefully — they contain the whole story of this dataset.

Applicants with **no checking account at all** default 11.7% of the time,
while those holding one with a balance under 100 DM default 49.3% of the
time. The counter-intuitive result — no account is *safer* — is real, and it
is the kind of thing that makes credit modelling interesting: having no
account at this bank often means banking elsewhere, which is not a risk
signal.

Credit history is stranger still. "No credits taken / all paid back duly"
shows the **highest** default rate. A borrower with no track record is not
safe — they are unmeasured. Thin-file applicants are a genuine industry
problem.

The next cell draws the same two tables as horizontal bar charts (`ax.barh`), with a dashed line at
the 30% portfolio average and the sample size printed beside each bar.

In [ ]:
# One figure with two side-by-side panels (1 row, 2 columns); figsize is width x height in inches.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Loop over (panel, column, title) triples so both panels are drawn by the same code.
for ax, column, title in [
    (axes[0], "status", "Checking account status"),  # left panel
    (axes[1], "credit_history", "Credit history"),  # right panel
]:
    # Default rate per category, re-sorted lowest first so the riskiest bar sits at the top.
    table = default_rate_by(column).sort_values("default_rate")
    # Category names truncated to 32 characters so the labels fit.
    labels = [str(i)[:32] for i in table.index]
    # Horizontal bar chart: one bar per category, length = default rate in %.
    ax.barh(labels, table["default_rate"], color="#2a78d6", height=0.65)
    # Dashed vertical line at the portfolio-wide default rate (30%) for reference.
    ax.axvline(credit["default"].mean() * 100, color="#eb6834", linewidth=1.5, linestyle="--")
    # Panel title.
    ax.set_title(title)
    # x-axis label.
    ax.set_xlabel("Default rate (%)")
    # Vertical grid lines only, to help read bar lengths...
    ax.grid(axis="x")
    # ...and no horizontal grid lines.
    ax.grid(axis="y", visible=False)
    # zip pairs each rate with its count; enumerate gives the bar position y = 0, 1, 2, ...
    for y, (rate, n) in enumerate(zip(table["default_rate"], table["applications"])):
        # Write "rate% (n=count)" just past the end of each bar — always show the denominator.
        ax.text(rate + 0.7, y, f"{rate:.0f}%  (n={n})", va="center", fontsize=8, color="#52514e")
    # Widen the x-axis by 35% so the text labels are not clipped.
    ax.set_xlim(0, table["default_rate"].max() * 1.35)

# Label the dashed average line on the left panel.
axes[0].annotate("portfolio average 30%", xy=(30, -0.55), fontsize=8,
                 color="#eb6834", ha="center")  # orange to match the line; centred on x = 30
# Figure-level title, left-aligned.
fig.suptitle("Default rate by category, with sample sizes", x=0.02, ha="left",
             fontsize=13, fontweight="bold")  # larger bold title text
# Tidy the spacing so labels do not overlap.
fig.tight_layout()
# Render the figure.
plt.show()

Note the `n=` labels. The two worst credit-history categories rest on 40 and
49 applications respectively, so those rates carry wide error bars. A high
default rate on a handful of applications is not a finding. Always show the
denominator.

Next, the three continuous variables. For each one the cell overlays two histograms: borrowers who
repaid (blue) and borrowers who defaulted (orange). `np.histogram_bin_edges` computes one shared set
of bins so the two histograms line up. `density=True` scales each to the same total area, so the
smaller defaulted group (300 loans) can be compared with the repaid group (700). Look for the orange
distribution sitting to the **right** on duration and amount, and to the **left** on age.

In [ ]:
# Continuous variables: do defaulters borrow more, for longer?
# Three side-by-side panels, one per variable.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Loop over (panel, column, axis label) triples.
for ax, column, label in [
    (axes[0], "duration", "Loan duration (months)"),  # left
    (axes[1], "amount", "Loan amount (DM)"),  # middle
    (axes[2], "age", "Applicant age (years)"),  # right
]:
    # .loc[row condition, column]: the values for borrowers who repaid (default == 0)...
    repaid = credit.loc[credit["default"] == 0, column]
    # ...and for borrowers who defaulted (default == 1).
    defaulted = credit.loc[credit["default"] == 1, column]
    # One shared set of 25 bin edges, so the two histograms are directly comparable.
    bins = np.histogram_bin_edges(credit[column], bins=25)
    # density=True scales each histogram to area 1, so groups of different size (700 vs 300) compare fairly.
    ax.hist(repaid, bins=bins, density=True, alpha=0.6, color="#2a78d6", label="Repaid")
    # alpha=0.6 makes the bars semi-transparent so the overlap stays visible.
    ax.hist(defaulted, bins=bins, density=True, alpha=0.6, color="#eb6834", label="Defaulted")
    # Axis labels.
    ax.set_xlabel(label)
    # The y-axis is a density, so label it loosely...
    ax.set_ylabel("Share of group")
    # ...and hide the tick numbers, which have no intuitive meaning.
    ax.set_yticks([])

# One legend is enough for all three panels.
axes[0].legend()
# Figure title stating the finding.
fig.suptitle("Defaulters borrow more, for longer, and are younger", x=0.02, ha="left",
             fontsize=13, fontweight="bold")  # larger bold title text
# Tidy spacing and draw.
fig.tight_layout()
# Render the figure.
plt.show()

# The same story in numbers: mean duration, amount and age for repaid (0) vs defaulted (1).
print(credit.groupby("default")[["duration", "amount", "age"]].mean().round(1).to_string())

## 3. Fair lending — the variables you must not use

This section is not optional and it is not a footnote. It is the part of
credit modelling most likely to end a career.

In the United States, the **Equal Credit Opportunity Act** (ECOA) and its
implementing **Regulation B** prohibit discrimination in any aspect of a
credit transaction on the basis of race, colour, religion, national origin,
**sex**, marital status, or age (for applicants old enough to contract), or
because income derives from public assistance.

This dataset contains three variables that are legally radioactive:

The next cell counts each category of the two protected variables with `value_counts()` and prints
the age range. Notice that `personal_status_sex` **combines sex and marital status** in one column,
and both are protected under ECOA.

In [ ]:
# value_counts() counts how many applications fall in each category, largest first.
print(credit["personal_status_sex"].value_counts().to_string())
# Blank line between the two tables.
print()
# Counts of foreign workers (yes) and others (no).
print(credit["foreign_worker"].value_counts().to_string())
# Youngest and oldest applicant: age is protected too, which matters for the discussion below.
print(f"\nAge range: {credit['age'].min()} to {credit['age'].max()}")

Now apply `default_rate_by` to the protected variables. Look at the **sample sizes** as well as the
rates. Only 37 applicants are non-foreign workers, so their 10.8% default rate is very imprecise.

In [ ]:
# What do those variables look like in the outcome data?
# Reuse the default_rate_by helper on each protected column.
for column in ["personal_status_sex", "foreign_worker"]:
    # Print the column name as a heading.
    print(f"\n{column}")
    # Applications and default rate (%) per category.
    print(default_rate_by(column).to_string())

There *are* differences in the raw default rates. That is exactly the trap.
A variable being statistically predictive does not make it lawful to use, and
under ECOA the intent behind the model is irrelevant — what matters is the
effect.

Two further points that are easy to miss:

- **Dropping the variable is not sufficient.** A model can reconstruct a
  protected characteristic from correlated variables — this is *proxy
  discrimination*, and it is the reason fair-lending testing looks at
  outcomes, not just inputs. Postal code standing in for race is the classic
  US example.
- **Disparate impact is actionable without disparate treatment.** A
  facially neutral rule that produces a significant adverse effect on a
  protected class can violate the Act even if nobody intended it.

For this lab we drop the two most obviously protected variables. `age` is a
harder call: it is protected under ECOA for applicants of legal contracting
age, but it is also a legitimate underwriting input in some jurisdictions
and product types. We keep it here and flag it, which is what an honest model
card would do.

#### Background: fairness and protected variables

**Intuition.** A credit model is a decision rule applied to people, so it has to be lawful as well as
accurate. A model learns any pattern that predicts the target. It cannot tell a legitimate risk
driver (missed payments) from a characteristic society has decided must not count (sex, national
origin).

**The legal framework (US).**

- **ECOA / Regulation B** list the prohibited bases: race, colour, religion, national origin, sex,
  marital status, age, and receipt of public assistance income.
- **Disparate treatment** means using a protected characteristic directly, by feeding it into the
  model or setting different rules by group.
- **Disparate impact** means a neutral-looking rule produces markedly worse outcomes for a protected
  group without a business justification that a less discriminatory alternative could not meet.
- **Adverse action notices** (ECOA and the Fair Credit Reporting Act) require the lender to give a
  declined applicant the principal reasons for the decision. The model's reasoning has to be
  extractable.

**How lenders test for it.**

- *Exclude* protected variables from the inputs. That is necessary but not sufficient.
- *Test outcomes by group.* A common screen is the **adverse impact ratio**: the approval rate of the
  protected group divided by that of the reference group. Values below about 0.8 (the "four-fifths
  rule" borrowed from employment law) trigger a closer look.
- *Look for proxies.* Check whether retained variables (postcode, occupation, even `purpose` or
  `housing`) can reconstruct the protected attribute. If a simple model predicts sex from the other
  inputs, the information is still in there.
- *Search for less discriminatory alternatives.* If a slightly different model is almost as accurate
  but has smaller group disparities, regulators expect you to prefer it.

**In finance.** Fair-lending exams by the CFPB, OCC and Federal Reserve. The EU AI Act classes
creditworthiness assessment as *high-risk*. Model risk management (SR 11-7) expects documented
variable selection, and the next cell is exactly that.

**This cell** puts the protected columns in a named list `PROTECTED`, builds `credit_model` without
them (and without the ID), and removes them from `categorical_columns`. Expect 18 features to remain.

In [ ]:
# The protected variables we refuse to use, kept in one named list so the decision is visible and auditable.
PROTECTED = ["personal_status_sex", "foreign_worker"]

# Modelling table: drop the ID column plus the protected columns (drop returns a new DataFrame; credit is untouched).
credit_model = credit.drop(columns=["applicant_id"] + PROTECTED)
# Rebuild the categorical list without the protected names (a list comprehension that filters them out).
categorical_columns = [c for c in categorical_columns if c not in PROTECTED]

# Record the decisions, as a model card would.
print("Dropped from the model:", ", ".join(PROTECTED))
# Age stays in, with the caveat documented.
print("Retained with a documented caveat: age")
# shape[1] is the number of columns; subtract 1 for the target, "default".
print(f"\nFeatures remaining: {credit_model.shape[1] - 1}")

### Exercise 1 (worked solution)

Train the model below twice — once as written, and once with
`personal_status_sex` added back in. Compare the ROC-AUC. If including the
protected variable improves the model, does that change your view about
whether it should be used? Write two or three sentences.

*Where the solution is:* the exercise needs the train/test split and the logistic-regression pipeline,
which are built in sections 4–6. The worked solution therefore sits straight after section 6, under
**Exercise 1 — solution code**.

## 4. Splitting the data

Unlike the market panel in Notebook 03, these are cross-sectional
observations with no time ordering, so a **random** split is correct here.

We stratify on the target so that the training and test sets have the same
30% default rate. Without stratification, a small test set can drift to 25%
or 35% by chance and every metric moves with it.

#### Background: scikit-learn and the estimator API

**scikit-learn** (`sklearn`) is Python's standard library for classical machine learning: regression,
classification, clustering, preprocessing and model evaluation. Banks, insurers and fintechs use it
for credit scoring, fraud flags and churn models. It is free, well tested and well documented, and
every algorithm shares **one interface**, so swapping logistic regression for a random forest is a
one-line change (section 9 does exactly that).

Everything in scikit-learn is an **estimator** object, used in two stages:

1. **Create** it with its *hyper-parameters*, the settings you choose: `LogisticRegression(max_iter=2000)`.
   Nothing has been learned yet.
2. **Fit** it on data: `model.fit(X_train, y_train)`. This learns the *parameters* from the data and
   stores them on the object as attributes ending in an underscore: `coef_`, `mean_`, `categories_`.

After fitting, which methods you call depends on the kind of estimator:

| Method | Who has it | What it returns |
|---|---|---|
| `fit(X, y)` | everything | the fitted estimator itself (learning happens here) |
| `predict(X)` | models | a hard label per row: 0 or 1 |
| `predict_proba(X)` | classifiers | an array of shape (rows, 2): $P(\text{class }0)$ and $P(\text{class }1)$ per row |
| `score(X, y)` | models | a default metric; **accuracy** for classifiers |
| `transform(X)` | preprocessors | the converted data (scaled, encoded) |
| `fit_transform(X)` | preprocessors | `fit` then `transform` in one call; only ever on training data |

The core discipline is that **anything learned comes from training rows only**. Test rows only ever
go through `predict`, `predict_proba`, `score` or `transform`.

#### Background: `train_test_split` with `stratify`

**Intuition.** A model always looks good on the data it learned from. To estimate how it will do on
*next month's* applicants, hide some applicants away and use them only to score the finished model.

**What it does.** `train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)` shuffles the
row positions with the given seed and puts 25% of them in the test set (250 rows) and 75% in training
(750 rows). It returns four objects, always in the order `X_train, X_test, y_train, y_test`.

**`stratify=y`.** Without it, the shuffle is purely random, and a 250-row test set might hold 65 or 85
defaulters instead of 75. Stratification splits each class separately (the 700 repaid and the 300
defaulted) in the same 75/25 proportion, so both sets keep the 30% default rate exactly. This matters
most when the minority class is small, as it usually is in credit (defaults), fraud and AML
(suspicious transactions).

**When a random split is wrong.** Rows here are independent applicants with no time order. With
time-ordered data such as prices or macro series, a random split lets the model learn from the future
and predict the past. That is look-ahead bias, and Notebook 05 uses a walk-forward split instead.

**This cell** separates the features `X` from the target `y`, splits them, and prints both default
rates. Look for **30.0%** in both.

In [ ]:
# train_test_split randomly divides rows into a training set and a test set.
from sklearn.model_selection import train_test_split

# X = the features (every column except the target) — 1,000 rows x 18 columns.
X = credit_model.drop(columns="default")
# y = the target: 1 = defaulted, 0 = repaid.
y = credit_model["default"]

# Hold back 25% of applications as a test set the model never sees while learning.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y  # stratify=y keeps 30% defaults in both parts
)

# Check: 750 training rows with the same 30% default rate...
print(f"Training set: {len(X_train)} applications, {y_train.mean():.1%} default")
# ...and 250 test rows with 30% defaults.
print(f"Test set:     {len(X_test)} applications, {y_test.mean():.1%} default")

## 5. Preprocessing as part of the model

The categorical columns hold text. Models need numbers. **One-hot encoding**
turns a column with $k$ categories into $k$ indicator columns.

The important design choice is to put the encoder *inside* a `Pipeline`.
Then `fit` on training data learns the categories and the scaling, and
`predict` on test data applies them — so the leak we had to be careful about
by hand in Notebook 03 becomes structurally impossible.

#### Background: `OneHotEncoder`

**Intuition.** Models do arithmetic, so the text `"no checking account"` must become numbers. Coding
the categories 1, 2, 3, 4 would invent an order and distances that do not exist. One-hot encoding
creates a separate yes/no (0/1) column for each category instead.

**Algorithm.** `fit` records the distinct categories in each column (`categories_`). `transform`
outputs, for every row, a 1 in the column of that row's category and 0 elsewhere. A `status` column
with 4 categories becomes 4 indicator columns.

**The arguments used here.**

- `drop="first"` drops one indicator per column, leaving $k-1$. The dropped category is the
  **reference group**, and every coefficient on that variable is read relative to it. Without the
  drop, the $k$ indicators always sum to 1, which duplicates the model's intercept (the "dummy
  variable trap").
- `handle_unknown="ignore"` means that if a category turns up in new data that was never seen in
  training, the row gets all zeros instead of raising an error. That is essential in production,
  where a new loan `purpose` code can appear any day.

**Weakness.** Many-valued columns (postcode, merchant ID) explode into thousands of sparse columns. Rare
categories get unreliable coefficients.

#### Background: `StandardScaler`

**Intuition.** Put every numeric feature on the same scale so that none dominates just because of its
units (DM versus months).

**Algorithm.** `fit` learns each column's mean $\mu$ and standard deviation $\sigma$ (`mean_`,
`scale_`). `transform` returns the **z-score** $z = (x-\mu)/\sigma$, so each column ends up with mean 0
and standard deviation 1. A loan amount of one standard deviation above average becomes `1.0`.

**Why fit on training only.** If $\mu$ and $\sigma$ were computed on all 1,000 rows, a little
information about the test applicants would leak into the training inputs. On this dataset the effect
is small, but with time series it can be serious, so the habit matters.

**Who needs it.** Regularised linear models (the logistic regression here) and distance-based
methods need it. Tree models do not. A split "amount ≤ 5,000" is the same split after any increasing
rescaling.

#### Background: `ColumnTransformer`

Real tables mix types. `ColumnTransformer` takes a list of `(name, transformer, columns)` triples,
applies each transformer to its own columns, and **concatenates the outputs side by side**. Here the 7
numeric columns go through `StandardScaler` and the 11 categorical columns through `OneHotEncoder`.
Any column not listed is dropped by default. `get_feature_names_out()` returns the output names
prefixed with the step name, such as `numeric__duration` and `categorical__status_no checking account`.

#### Background: `Pipeline` (and why it prevents leakage)

**Intuition.** A `Pipeline` is an assembly line. It chains any number of transformers with a model at
the end and behaves as **one estimator**.

**Algorithm.**

- `pipeline.fit(X_train, y_train)` calls `fit_transform` on each step in turn, passing the output to
  the next, then `fit` on the final model.
- `pipeline.predict(X_test)` calls only `transform` on each step, using what was learned in training,
  then `predict`.
- Steps are named, and `pipeline.named_steps["model"]` retrieves one.

**Why it prevents leakage.** There is no way to call the pipeline that fits the scaler or encoder on
test data. Under cross-validation (later) the whole pipeline is refitted inside each fold, so the
preprocessing never sees the held-out fold either. In industry the fitted pipeline is also the unit
you **deploy**. Raw application data goes in, a probability comes out, and the production
preprocessing cannot drift from the training preprocessing.

**This cell** defines the `ColumnTransformer` and fits it alone on `X_train`, just to show what it
produces. Look for **18 original columns becoming 44 model columns** and for the prefixed names.

In [ ]:
# ColumnTransformer applies different preprocessing to different columns and glues the results side by side.
from sklearn.compose import ColumnTransformer
# Pipeline chains preprocessing and a model into a single estimator with one fit/predict.
from sklearn.pipeline import Pipeline
# OneHotEncoder turns text categories into 0/1 columns; StandardScaler turns numbers into z-scores.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define (but do not yet fit) the preprocessing step.
preprocessor = ColumnTransformer(
    transformers=[  # list of (name, transformer, columns) triples
        ("numeric", StandardScaler(), numeric_columns),  # (name, transformer, columns): z-score the 7 numeric columns
        (  # second triple, spread over several lines to fit the comment
            "categorical",  # name of the second step
            # handle_unknown="ignore" stops the model crashing when a category
            # appears in production that was not in the training data.
            # drop="first" drops one category per column as the reference group (avoids redundant columns).
            OneHotEncoder(handle_unknown="ignore", drop="first"),
            categorical_columns,  # the 11 remaining text columns
        ),
    ]
)

# Fit on the TRAINING rows only: learns each numeric mean/std and the list of categories per column.
preprocessor.fit(X_train)
# The names of the output columns, prefixed with the step name ("numeric__", "categorical__").
encoded_names = preprocessor.get_feature_names_out()
# 18 input columns become 44 model columns after one-hot encoding.
print(f"{X_train.shape[1]} original columns became {len(encoded_names)} model columns")
# Heading for the sample list.
print("\nFirst twelve:")
# Print the first twelve output column names, one per line.
for name in encoded_names[:12]:
    # Indented for readability.
    print("   ", name)

## 6. Logistic regression

The workhorse of credit scoring, and still the model most likely to be in
production at a regulated lender — because you can explain it. Under the
Fair Credit Reporting Act a declined applicant is entitled to the specific
reasons for the decision, and "the gradient boosting ensemble said no" is not
a reason.

Logistic regression models the log-odds of default as a linear function of
the features:

$$\ln\!\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k$$

#### Background: logistic regression

**Intuition.** We want a probability of default between 0 and 1, yet a linear score
$z = \beta_0 + \beta_1x_1 + \dots$ can be any number. Logistic regression passes the score through
the **sigmoid** (logistic) function, which squashes any real number into (0, 1):

$$p = \sigma(z) = \frac{1}{1+e^{-z}}$$

A score of 0 gives $p = 0.5$, large positive scores give $p$ near 1, and large negative scores give
$p$ near 0. Inverting the sigmoid gives the **log-odds** equation above. Each feature shifts the
log-odds linearly, so $e^{\beta_j}$ is the **odds ratio** for a one-unit increase in $x_j$ (section 8).

**How the coefficients are found: maximum likelihood.** Choose the $\beta$ values that make the
observed outcomes most probable. Equivalently, minimise the **log-loss**

$$-\sum_i \big[\,y_i \ln p_i + (1-y_i)\ln(1-p_i)\,\big]$$

which punishes confident wrong predictions very heavily. There is no closed-form solution, so an
iterative optimiser (scikit-learn's default is `lbfgs`) walks downhill until the improvement is tiny.

**Regularisation.** scikit-learn's `LogisticRegression` adds an **L2 penalty** $\tfrac{1}{2C}\sum\beta_j^2$
by default, with `C=1.0`. The penalty shrinks coefficients towards zero, which tames overfitting when
there are many dummy columns and few rows. A smaller `C` means stronger shrinkage. Because the penalty
treats all coefficients alike, the features must be on comparable scales, which is another reason for
`StandardScaler`.

**Hyper-parameters in this code.**

- `max_iter=2000` lets the optimiser take up to 2,000 steps. The default of 100 can stop early with a
  convergence warning.
- `random_state` fixes any randomness in the solver.
- `C` is left at its default of 1.0.

**Strengths.** Fast, stable and well calibrated, and the coefficients explain the decision: one odds
ratio per reason, ready for an adverse-action notice. **Weaknesses.** It only captures interactions
and curves that you build into the features yourself. **In finance:** the classic credit scorecard
(points = scaled coefficients), probability-of-default (PD) models for Basel capital, and the
baseline every fraud or churn model must beat.

#### Background: accuracy and the base-rate problem

`pipeline.score(X, y)` returns **accuracy**: the share of rows where `predict` (cut-off 0.5) matches
the truth. With imbalanced classes, accuracy has a built-in floor. A "model" that **always predicts
the majority class** (approve everyone) scores $1 - \text{default rate}$, which is 0.70 here without
looking at a single feature. In fraud (0.1% positive) the trivial model is 99.9% accurate and
catches nothing. Always print the base rate next to accuracy, as this cell does.

**This cell** builds the pipeline (the preprocessor followed by `LogisticRegression`), fits it on the
training rows, and prints training accuracy, test accuracy and the approve-everyone benchmark. Look
for test accuracy of **0.776** against a benchmark of **0.700**.

In [ ]:
# scikit-learn's logistic regression classifier.
from sklearn.linear_model import LogisticRegression

# A two-step pipeline: preprocess the raw columns, then fit the logistic model on the 44 encoded columns.
logistic = Pipeline(
    steps=[  # list of (name, step) pairs, run in order
        ("preprocess", preprocessor),  # step 1: encoding and scaling
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),  # step 2: max_iter gives the optimiser room to converge
    ]
)

# fit runs the steps in order on the training rows: the preprocessor learns, transforms, then the model learns coefficients.
logistic.fit(X_train, y_train)

# score() on a classifier returns accuracy: the share of rows predicted correctly at the 0.5 cut-off.
train_accuracy = logistic.score(X_train, y_train)
# The same on the unseen test rows — the honest number.
test_accuracy = logistic.score(X_test, y_test)
# Three decimal places.
print(f"Training accuracy: {train_accuracy:.3f}")
# Test accuracy.
print(f"Test accuracy:     {test_accuracy:.3f}")
# The base rate: a "model" that approves everyone is right on every non-defaulter, i.e. 1 - default rate.
print(f"\nAlways predicting 'no default' would score: {1 - y_test.mean():.3f}")

Look hard at that last line. The model is barely beating the strategy of
approving every single application. Accuracy is a terrible metric on
imbalanced data, and credit data is always imbalanced. We need better
measures — and we need them before we believe anything.

### Exercise 1 — solution code

Now that the split and the logistic pipeline exist, we can answer Exercise 1 (stated in section 3).
The cell builds a second modelling table that keeps `personal_status_sex` (still dropping
`foreign_worker` and the ID) and splits it with exactly the same settings, so the test applicants are
the same 250 people. It then fits an otherwise identical logistic pipeline and compares test ROC-AUC.
`roc_auc_score` is introduced properly in section 7. For now, read it as "how well the model ranks
defaulters above repayers; 0.5 is random, 1.0 is perfect".

Because a single 250-row test set is noisy, the cell also compares the two models with **5-fold
stratified cross-validation**, explained fully in the cross-validation section. `clone(logistic)`
makes an unfitted copy of the section-6 pipeline so the original stays as it is.

In [ ]:
# Exercise 1 — does adding personal_status_sex back in improve the model?
# roc_auc_score is formally imported in section 7; import it here too so this cell runs where it sits.
from sklearn.metrics import roc_auc_score
# clone() makes an unfitted copy of an estimator with the same settings.
from sklearn.base import clone
# The cross-validation tools from the "Cross-validation" section, imported early for the same reason.
from sklearn.model_selection import StratifiedKFold, cross_val_score

# A modelling table that keeps personal_status_sex but still drops the ID and foreign_worker. New names: nothing later is overwritten.
credit_with_sex = credit.drop(columns=["applicant_id", "foreign_worker"])
# Features: 19 columns this time.
X_sex = credit_with_sex.drop(columns="default")
# The same target.
y_sex = credit_with_sex["default"]

# Identical split settings (same seed, same stratification, same row count) select exactly the same applicants.
X_sex_train, X_sex_test, y_sex_train, y_sex_test = train_test_split(
    X_sex, y_sex, test_size=0.25, random_state=RANDOM_STATE, stratify=y_sex  # same arguments as section 4
)
# Confirm the test applicants are the same people, so the comparison is like for like.
print("Same test applicants as the main model:", X_sex_test.index.equals(X_test.index))

# A second preprocessor whose categorical list has personal_status_sex added back.
preprocessor_sex = ColumnTransformer(
    transformers=[  # list of (name, transformer, columns) triples
        ("numeric", StandardScaler(), numeric_columns),  # numeric columns unchanged
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"),  # same encoder settings as section 5
         categorical_columns + ["personal_status_sex"]),  # the 11 permitted columns plus the protected one
    ]
)
# The same logistic regression, with the extended preprocessor in front.
logistic_sex = Pipeline(
    steps=[  # list of (name, step) pairs, run in order
        ("preprocess", preprocessor_sex),  # step 1
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),  # step 2, same settings as section 6
    ]
)
# Fit on the training rows only.
logistic_sex.fit(X_sex_train, y_sex_train)

# Test-set ROC-AUC of the model as written (fitted in section 6).
auc_without = roc_auc_score(y_test, logistic.predict_proba(X_test)[:, 1])
# Test-set ROC-AUC with personal_status_sex included.
auc_with = roc_auc_score(y_sex_test, logistic_sex.predict_proba(X_sex_test)[:, 1])

# Print both and the difference.
print(f"\nTest ROC-AUC without personal_status_sex: {auc_without:.3f}")
# With the protected variable.
print(f"Test ROC-AUC with personal_status_sex:    {auc_with:.3f}")
# "+.3f" always shows the sign.
print(f"Difference:                               {auc_with - auc_without:+.3f}")

# One 250-row test set is noisy, so repeat the comparison with 5-fold cross-validation on all 1,000 rows.
folds_ex1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# clone(logistic) gives an unfitted copy of the section-6 pipeline; cross_val_score returns 5 AUCs.
cv_without = cross_val_score(clone(logistic), X, y, cv=folds_ex1, scoring="roc_auc")
# The same five folds (same seed, same rows) for the model with the protected variable.
cv_with = cross_val_score(logistic_sex, X_sex, y_sex, cv=folds_ex1, scoring="roc_auc")
# Mean ± standard deviation across folds for each.
print(f"\n5-fold CV ROC-AUC without: {cv_without.mean():.3f} ± {cv_without.std():.3f}")
# With.
print(f"5-fold CV ROC-AUC with:    {cv_with.mean():.3f} ± {cv_with.std():.3f}")
# Fold-by-fold gain from adding the variable.
print(f"Gain in each fold:         {np.round(cv_with - cv_without, 3)}")

**Discussion.** Adding `personal_status_sex` made almost no difference. On the test set the ROC-AUC
moved from **0.800 to 0.799**, very slightly worse. Under 5-fold cross-validation it moved from
**0.778 to 0.783**, a gain of about 0.005, against a fold-to-fold standard deviation of about 0.019.
The per-fold gains were all small, between 0.001 and 0.008. So on this data the protected variable
adds essentially nothing that the other features do not already carry, which is itself a hint of
proxy effects.

The size of the gain does not change the answer. Even if it had added 0.05 of AUC, the variable
should not be used. ECOA and Regulation B prohibit credit decisions based on sex or marital status
regardless of predictive value. Using it would be disparate treatment, which is illegal whatever the
intent and however good the statistics.

## 7. Evaluating a classifier properly

The **confusion matrix** is the foundation. Every other metric is a ratio
computed from its four cells.

#### Background: the confusion matrix

**Intuition.** A classifier can be right in two ways and wrong in two ways. The confusion matrix
counts all four, with **rows = what actually happened** and **columns = what the model predicted**.
"Positive" means the event we are trying to catch, a default.

| | Predicted repay (0) | Predicted default (1) |
|---|---|---|
| **Actually repaid (0)** | True negative (TN): good loan approved | False positive (FP): good loan declined |
| **Actually defaulted (1)** | False negative (FN): bad loan approved | True positive (TP): bad loan caught |

**The two errors cost different amounts.** A false negative loses principal (loss given default × the
amount). A false positive only loses the margin we would have earned. Section 10 turns that asymmetry
into a cut-off.

**This cell** imports the metric functions and computes:

- `y_pred = logistic.predict(X_test)`: hard 0/1 labels, using an implicit 0.5 cut-off.
- `y_proba = logistic.predict_proba(X_test)[:, 1]`: the probability of default for every test
  applicant. `[:, 1]` selects the second column, class 1.
- `confusion_matrix(y_test, y_pred)`: a 2×2 array. `.ravel()` flattens it into the order
  **TN, FP, FN, TP**.

Look for **35 false negatives** (bad loans approved) against **40 true positives**.

In [ ]:
# Import the evaluation tools used in this section.
from sklearn.metrics import (
    ConfusionMatrixDisplay,  # draws a confusion matrix as a coloured grid
    classification_report,  # precision, recall and F1 per class in one table
    confusion_matrix,  # the 2x2 table of counts
    roc_auc_score,  # area under the ROC curve
    roc_curve,  # the points of the ROC curve
    precision_recall_curve,  # the points of the precision-recall curve
    average_precision_score,  # area under the precision-recall curve
)

# predict() returns hard 0/1 labels, using a 0.5 probability cut-off.
y_pred = logistic.predict(X_test)
# predict_proba() returns a (250, 2) array of [P(repay), P(default)]; [:, 1] keeps the default probability.
y_proba = logistic.predict_proba(X_test)[:, 1]

# confusion_matrix(actual, predicted) gives [[TN, FP], [FN, TP]] with rows = actual, columns = predicted.
matrix = confusion_matrix(y_test, y_pred)
# ravel() flattens the 2x2 into four numbers in that order, unpacked into named variables.
true_negative, false_positive, false_negative, true_positive = matrix.ravel()

# Print the table by hand with labelled rows and columns.
print("                        Predicted repay    Predicted default")
# ">16" right-aligns the number in a 16-character field so the columns line up.
print(f"Actually repaid  {true_negative:>16}  {false_positive:>18}")
# Second row: defaulters we approved (FN) and defaulters we caught (TP).
print(f"Actually defaulted{false_negative:>15}  {true_positive:>18}")
# Blank line.
print()
# False negatives: approved loans that went bad — lost principal.
print(f"The {false_negative} bad loans we approved are the expensive mistake.")
# False positives: declined loans that would have repaid — lost interest.
print(f"The {false_positive} good loans we declined cost us foregone margin.")

The same matrix drawn as a picture with `ConfusionMatrixDisplay`. Darker blue means more loans in
that cell. The heavy diagonal (154 and 40) is correct decisions. The off-diagonal cells (21 and 35)
are the two kinds of mistake.

In [ ]:
# A single 5.5 x 4.5 inch panel.
fig, ax = plt.subplots(figsize=(5.5, 4.5))
# Wrap the matrix computed above in a display object with readable class labels...
ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=["Repaid", "Defaulted"]  # the counts, and names for rows/columns 0 and 1
).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")  # ...and draw it: blue shading, integer ("d") counts
# Title records the cut-off used.
ax.set_title("Logistic regression, default 0.5 cut-off")
# Grid lines would cut through the cells, so turn them off.
ax.grid(False)
# Render the figure.
plt.show()

#### Background: precision, recall and F1

All three are ratios of confusion-matrix cells, computed separately for each class. For the
**Defaulted** class:

- **Precision** $= \dfrac{TP}{TP+FP}$. Of the applicants we flagged as risky, the share who really
  defaulted. Low precision means turning away a lot of good customers.
- **Recall** (sensitivity, true positive rate) $= \dfrac{TP}{TP+FN}$. Of the borrowers who really
  defaulted, the share we caught. Low recall means bad loans slipping through.
- **F1** $= 2\cdot\dfrac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$, the
  harmonic mean. It is high only when both are high.

There is a **trade-off**. Lowering the cut-off flags more applicants, which raises recall and usually
lowers precision. In fraud and AML monitoring the same tension appears as "alerts the investigators
can handle" against "fraud that gets through".

`classification_report` prints these for both classes. `support` is the number of actual cases in
each class (175 repaid, 75 defaulted). The *macro avg* is the simple average of the two classes and
the *weighted avg* weights them by support. Look at **recall for Defaulted: 0.53**. The model catches
only about half the bad loans at the 0.5 cut-off. Check it by hand: 40 / (40 + 35) = 0.533.

In [ ]:
# Precision, recall, F1 and support (row count) for each class, plus overall averages.
print(classification_report(y_test, y_pred, target_names=["Repaid", "Defaulted"]))

The vocabulary, in lending terms:

| Metric | Formula | What it means to a lender |
|---|---|---|
| **Precision** (defaulted) | TP / (TP + FP) | of the applications we declined for risk, how many would really have gone bad |
| **Recall** (defaulted) | TP / (TP + FN) | of the loans that would have gone bad, how many did we catch |
| **F1** | harmonic mean | a single number when you must have one |

Recall on the defaulted class is the number that matters most here, and it is
low. The model is missing most of the bad loans. We will fix that in section
10 — not by changing the model, but by changing the cut-off.

### ROC-AUC: the threshold-free measure

The ROC curve traces the trade-off between catching bad loans (true positive
rate) and wrongly declining good ones (false positive rate) across **every**
possible cut-off. The area under it is the probability that the model gives a
randomly chosen defaulter a higher score than a randomly chosen repayer.

0.5 is a coin flip. In credit scoring, 0.70–0.80 is a usable model and above
0.85 you should check for leakage.

#### Background: the ROC curve and AUC

**Algorithm.** Sort the applicants by predicted default probability. Then sweep a cut-off from 1 down
to 0, and at each value compute:

- the **true positive rate** $TPR = TP/(TP+FN)$, which is recall, the bad loans caught;
- the **false positive rate** $FPR = FP/(FP+TN)$, the good loans wrongly declined.

Plotting TPR against FPR gives the ROC curve, running from (0, 0) where we decline nobody to (1, 1)
where we decline everybody. A useless model follows the diagonal. A perfect model goes straight up
the left edge.

**AUC** is the area under that curve. It has a neat interpretation: the probability that a randomly
chosen defaulter receives a **higher** score than a randomly chosen good borrower. It depends only on
the *ranking* of the scores, not on any cut-off or on calibration. Credit risk teams often quote the
**Gini coefficient** $= 2\cdot AUC - 1$ instead, which is the same information on a 0-to-1 scale.

**Accuracy versus ROC-AUC.** Accuracy judges one cut-off (0.5) and is inflated by the majority class.
AUC judges the ranking over *all* cut-offs and is unaffected by the base rate. A model can have mediocre
accuracy but a good AUC: it ranks applicants well, and you then choose the cut-off from the economics
(section 10).

**The precision–recall curve** (right panel) plots precision against recall as the cut-off moves. Its
baseline is the default rate (0.30), not 0.5, and its area is the **average precision (AP)**. When
positives are rare, as in fraud and AML, it is more informative than ROC.

**This cell** computes `roc_auc_score` and `average_precision_score`, gets the curve points from
`roc_curve` and `precision_recall_curve`, and plots both. Look for an **AUC of about 0.800** and an AP
well above the 0.30 baseline.

In [ ]:
# ROC-AUC uses the probabilities, not the 0/1 labels, so it judges the ranking across all cut-offs.
auc = roc_auc_score(y_test, y_proba)
# roc_curve returns false-positive rates, true-positive rates and the thresholds (discarded with "_").
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, y_proba)

# The precision and recall achieved at every possible threshold.
precision, recall, _ = precision_recall_curve(y_test, y_proba)
# Average precision summarises that curve as one number (its area).
average_precision = average_precision_score(y_test, y_proba)

# Two panels side by side.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left panel: the ROC curve, labelled with its AUC.
axes[0].plot(false_positive_rate, true_positive_rate, color="#2a78d6",
             label=f"Logistic regression (AUC {auc:.3f})")  # legend text includes the AUC
# The diagonal: what random scoring would achieve.
axes[0].plot([0, 1], [0, 1], color="#52514e", linestyle="--", linewidth=1,
             label="Random (AUC 0.500)")  # legend text for the diagonal
# x-axis: share of good loans wrongly declined.
axes[0].set_xlabel("False positive rate — good loans declined")
# y-axis: share of bad loans caught.
axes[0].set_ylabel("True positive rate — bad loans caught")
# Title.
axes[0].set_title("ROC curve")
# Legend in the empty bottom-right corner.
axes[0].legend(loc="lower right")

# Right panel: precision against recall.
axes[1].plot(recall, precision, color="#2a78d6",
             label=f"Logistic regression (AP {average_precision:.3f})")  # legend text includes the AP
# Horizontal line at the base rate (0.30): the precision of random guessing.
axes[1].axhline(y_test.mean(), color="#52514e", linestyle="--", linewidth=1,
                label=f"Base rate ({y_test.mean():.3f})")  # legend text for the baseline
# x-axis label.
axes[1].set_xlabel("Recall — share of bad loans caught")
# y-axis label.
axes[1].set_ylabel("Precision")
# Title.
axes[1].set_title("Precision–recall curve")
# Legend top right.
axes[1].legend(loc="upper right")

# Tidy spacing.
fig.tight_layout()
# Render the figure.
plt.show()

## 8. Reading the coefficients

This is what logistic regression buys you that a forest does not. Exponentiate
a coefficient and you get an **odds ratio**: the multiplicative effect on the
odds of default of a one-unit increase in that feature, holding everything
else fixed.

**The code.** `logistic.named_steps["preprocess"].get_feature_names_out()` returns the 44 column names
and `logistic.named_steps["model"].coef_[0]` returns the 44 matching coefficients. Both come from the
already-fitted pipeline. `np.exp` turns each coefficient into an odds ratio: above 1 raises the odds
of default, below 1 lowers them. `nlargest` and `nsmallest` pick the extremes. Look for
`status_no checking account` with an odds ratio of about **0.22**: such applicants have roughly a
fifth of the odds of default of the reference group (balance under 100 DM), all else equal.

In [ ]:
# Build a table pairing each model column name with its fitted coefficient.
coefficients = pd.DataFrame(
    {  # a dict: column name -> values
        "feature": logistic.named_steps["preprocess"].get_feature_names_out(),  # named_steps fetches a pipeline step by name
        "coefficient": logistic.named_steps["model"].coef_[0],  # coef_ is a (1, 44) array; [0] takes the single row
    }
)
# exp(coefficient) converts a log-odds effect into an odds ratio.
coefficients["odds_ratio"] = np.exp(coefficients["coefficient"])
# Strip the "numeric__" / "categorical__" prefixes so the names are readable (regex=False: plain text replace).
coefficients["feature"] = (
    coefficients["feature"].str.replace("numeric__", "", regex=False)  # .str applies a text method to every row
    .str.replace("categorical__", "", regex=False)  # chained: then remove the second prefix
)

# Heading.
print("Increases the odds of default the most:")
# nlargest(8, "coefficient") picks the 8 most positive coefficients; index=False hides row numbers.
print(coefficients.nlargest(8, "coefficient")[["feature", "coefficient", "odds_ratio"]].round(3).to_string(index=False))
# Heading.
print("\nDecreases the odds of default the most:")
# nsmallest picks the 8 most negative coefficients (odds ratios below 1).
print(coefficients.nsmallest(8, "coefficient")[["feature", "coefficient", "odds_ratio"]].round(3).to_string(index=False))

The twenty largest coefficients in absolute terms, as a bar chart. **Orange** bars (positive) raise
default risk and **blue** bars (negative) lower it. Bar length is in log-odds, so a bar of length 0.69
doubles or halves the odds, because $e^{0.69} \approx 2$.

In [ ]:
# Stack the 10 most positive and 10 most negative coefficients into one table.
top = pd.concat([coefficients.nlargest(10, "coefficient"), coefficients.nsmallest(10, "coefficient")])
# Sort so the chart runs from most negative (bottom) to most positive (top).
top = top.sort_values("coefficient")

# One tall panel.
fig, ax = plt.subplots(figsize=(9, 8))
# Orange for risk-increasing (positive) coefficients, blue for risk-reducing ones.
colors = ["#eb6834" if c > 0 else "#2a78d6" for c in top["coefficient"]]
# Horizontal bars; feature names cut to 44 characters so they fit.
ax.barh([f[:44] for f in top["feature"]], top["coefficient"], color=colors, height=0.7)
# Vertical line at zero: no effect.
ax.axvline(0, color="#52514e", linewidth=1)
# Title explains the colour code.
ax.set_title("Logistic regression coefficients — orange raises default risk, blue lowers it")
# Units are log-odds.
ax.set_xlabel("Coefficient (log-odds)")
# Vertical grid lines only.
ax.grid(axis="x")
# No horizontal grid lines.
ax.grid(axis="y", visible=False)
# Tidy spacing.
fig.tight_layout()
# Render the figure.
plt.show()

The numeric coefficients are on standardised features, so they read as "one
standard deviation more of this". The categorical ones are relative to the
dropped reference category.

A worked example: an odds ratio of 1.8 on a category means applicants in that
category have 1.8 times the odds of defaulting compared with the reference
group, all else equal. That sentence is exactly what goes in an adverse
action notice, which is why this model is still in production everywhere.

## 9. Tree-based models

Trees split the data on one variable at a time, recursively. They capture
interactions and non-linearities that a linear model cannot, and they are
invariant to feature scaling.

#### Background: decision trees

**Intuition.** A tree is a sequence of yes/no questions, the kind a loan officer might ask: "Does the
applicant have a checking account? Is the loan longer than 12 months?" Each answer sends the
applicant down a branch, and every **leaf** at the bottom predicts the default rate of the training
applicants who ended up there.

**Algorithm (CART, greedy recursive splitting).** At each node, try every feature and every possible
split point. Pick the split that makes the two child groups as **pure** as possible, measured by
**Gini impurity**:

$$G = 1 - \sum_k p_k^2 = 2p(1-p) \text{ for two classes}$$

where $p$ is the default share in the node. A node that is all one class has $G = 0$. A 50/50 node has
the maximum, $G = 0.5$. The chosen split maximises the drop in size-weighted impurity. The process
then repeats inside each child until a stopping rule is hit.

**Hyper-parameters here.**

- `max_depth=4` allows at most 4 questions in a row, which means at most 16 leaves.
- `min_samples_leaf=20` means every leaf must hold at least 20 training applicants, so no rule is
  built on a handful of cases.

Both are **regularisation**. An unrestricted tree keeps splitting until every leaf is pure, memorising
the training data.

**Strengths.** Readable, handles interactions and non-linearity automatically, needs no scaling.
**Weaknesses.** High variance: a slightly different sample can produce a very different tree. Accuracy
is usually the weakest of the four models. **In finance:** segmentation before building scorecards,
underwriting policy rules, and as the building block of the two ensembles below.

#### Background: random forests

**Intuition.** One tree is unstable, but the average of many different trees is stable: the wisdom
of crowds. The trees must make *different* mistakes for averaging to help, so the forest deliberately
injects randomness.

**Algorithm.**

1. **Bagging** (bootstrap aggregating): each tree is trained on a bootstrap sample, $n$ rows drawn
   *with replacement* from the training data.
2. **Feature subsampling**: at each split a tree considers only a random subset of features (the
   square root of the number of features by default for classification). That stops every tree from
   splitting first on the same dominant feature.
3. **Predict** by averaging the trees' predicted probabilities.

**Hyper-parameters here.**

- `n_estimators=400` grows 400 trees. More trees never hurt accuracy, they only cost time.
- `min_samples_leaf=5` makes each leaf hold at least 5 rows. That limits how far each tree can
  memorise and smooths the probabilities.
- `n_jobs=-1` builds the trees in parallel on every CPU core.
- `random_state` fixes the bootstrap draws.

**Strengths.** Strong performance with little tuning, robust to outliers, and hard to overfit badly.
**Weaknesses.** 400 trees cannot be read, and a reason for a decline has to come from post-hoc tools
(permutation importance, SHAP). **In finance:** fraud detection, AML transaction monitoring,
challenger models that benchmark the production scorecard.

#### Background: gradient boosting

**Intuition.** A random forest builds independent trees in parallel and averages them. **Boosting**
builds small trees **in sequence**, and each new tree concentrates on what the trees so far still get
wrong.

**Algorithm.**

1. Start with a constant prediction, the overall log-odds of default.
2. For each round $m = 1, \dots, M$:
   - compute each row's **gradient** of the log-loss, which for this loss is simply $y_i - p_i$ (the
     residual);
   - fit a small regression tree to those residuals;
   - add the tree's output to the running score, scaled by the **learning rate** $\eta$:
     $F_m(x) = F_{m-1}(x) + \eta\, h_m(x)$.
3. The final probability is $\sigma(F_M(x))$.

**Hyper-parameters.** This code uses scikit-learn's defaults:

- `n_estimators=100` rounds;
- `learning_rate=0.1`, the shrinkage that makes each tree a small correction;
- `max_depth=3` for each tree.

A smaller learning rate needs more trees but usually generalises better. Too many rounds overfit,
because the model starts fitting noise in the residuals.

**Strengths.** Often the most accurate method on tabular data. XGBoost, LightGBM and CatBoost are the
industry's go-to implementations. **Weaknesses.** More hyper-parameters to tune, easy to overfit on
small data, and opaque like the forest. **In finance:** card fraud scoring, marketing and collections
models, and increasingly credit underwriting at fintechs, paired with explainability tools.

**This cell** puts each of the four models behind the *same* preprocessor in a `Pipeline`, fits it on
the training set, and records training accuracy, test accuracy, test ROC-AUC and average precision.
Compare the `test_auc` column across models, and compare `train_accuracy` with `test_accuracy` within
each row.

In [ ]:
# A single decision tree, and plot_tree to draw one.
from sklearn.tree import DecisionTreeClassifier, plot_tree
# Two tree ensembles: a random forest (bagging) and gradient boosting (sequential trees).
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Dictionary of model name -> unfitted estimator, so one loop can train and score them all.
models = {
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),  # the baseline from section 6
    "Decision tree (depth 4)": DecisionTreeClassifier(  # a single, deliberately shallow tree
        max_depth=4, min_samples_leaf=20, random_state=RANDOM_STATE  # at most 4 levels of questions; every leaf holds >= 20 applicants
    ),
    "Random forest": RandomForestClassifier(  # an average of many randomised trees
        n_estimators=400, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1  # 400 trees, leaves >= 5 rows, n_jobs=-1 uses all CPU cores
    ),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),  # defaults: 100 trees of depth 3, learning_rate 0.1
}

# One dict of scores per model will be appended here.
results = []
# The fitted pipelines, kept by name for later sections.
fitted = {}

# .items() yields (name, estimator) pairs.
for name, estimator in models.items():
    # Wrap each model in the same preprocessing pipeline, so every model sees identical inputs.
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    # Fit preprocessing and model on the training rows only.
    pipeline.fit(X_train, y_train)
    # Keep the fitted pipeline for later use.
    fitted[name] = pipeline

    # Predicted default probability for each test applicant.
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    # Record four metrics for this model.
    results.append(
        {  # one dict = one future row of the results table
            "model": name,  # label
            "train_accuracy": pipeline.score(X_train, y_train),  # accuracy on rows it learned from
            "test_accuracy": pipeline.score(X_test, y_test),  # accuracy on unseen rows
            "test_auc": roc_auc_score(y_test, probabilities),  # ranking quality on unseen rows
            "avg_precision": average_precision_score(y_test, probabilities),  # precision-recall area on unseen rows
        }
    )

# Turn the list of dicts into a table indexed by model name, rounded to 3 decimals.
results_table = pd.DataFrame(results).set_index("model").round(3)
# Display it.
results_table

Look at the gap between `train_accuracy` and `test_accuracy`. A random forest
will typically score near-perfectly on the training data and much worse on
the test data. That gap is **overfitting**: the model has memorised the
training applications rather than learned a rule that generalises.

On a dataset with 1,000 rows and this much noise, the elaborate models rarely
beat logistic regression by enough to justify losing explainability. That is
a genuine and slightly unfashionable result, and it is why credit scorecards
still look the way they do.

*Reading this run's numbers.* With `min_samples_leaf=5` the forest is held back from memorising, so
its training accuracy is 0.856, not near 1.0. The largest train–test gap here belongs to **gradient
boosting** (0.901 against 0.768). Logistic regression shows almost no gap (0.768 against 0.776),
which is what a simple, regularised model looks like. On test AUC the four models span only 0.704 to
0.819.

The next cell draws training and test accuracy as paired bars, with the approve-everyone line at
0.70.

In [ ]:
# One panel for the train-versus-test comparison.
fig, ax = plt.subplots(figsize=(9, 4.5))
# Bar positions 0, 1, 2, 3 — one per model.
positions = np.arange(len(results_table))
# Light bars: training accuracy, nudged 0.2 up so the pair sits side by side.
ax.barh(positions - 0.2, results_table["train_accuracy"], height=0.36,
        color="#9ec5f4", label="Training accuracy")  # light blue, with legend label
# Dark bars: test accuracy, nudged 0.2 the other way.
ax.barh(positions + 0.2, results_table["test_accuracy"], height=0.36,
        color="#2a78d6", label="Test accuracy")  # dark blue, with legend label
# Label each position with the model name.
ax.set_yticks(positions, results_table.index)
# Dashed line at the "approve everyone" accuracy (0.70): the bar every model must clear.
ax.axvline(1 - y_test.mean(), color="#eb6834", linestyle="--", linewidth=1.5)
# Label that line; xytext/textcoords shift the text 4 points right of the anchor.
ax.annotate("approve everyone", xy=(1 - y_test.mean(), len(results_table) - 0.45),
            xytext=(4, 0), textcoords="offset points", fontsize=9, color="#eb6834")  # small orange text
# Axis label.
ax.set_xlabel("Accuracy")
# Show the full 0-1 scale so differences are not exaggerated.
ax.set_xlim(0, 1.05)
# Title states the lesson.
ax.set_title("The gap between the two bars is overfitting")
# Vertical grid lines only.
ax.grid(axis="x")
# No horizontal grid lines.
ax.grid(axis="y", visible=False)
# Legend bottom right.
ax.legend(loc="lower right")
# Render the figure.
plt.show()

### Seeing inside a tree

A shallow decision tree is the one machine learning model you can hand to a
credit committee and have them read it.

**`plot_tree`** draws a fitted `DecisionTreeClassifier`. Each box shows:

- the split rule, for example `status_no checking account <= 0.5`. Because the feature is a 0/1 dummy,
  "≤ 0.5" means *the applicant does **not** have that category*, and the left branch is always "rule
  true";
- `samples`, the number of training applicants reaching the box;
- `value`, the class counts;
- `class`, the majority class.

The colour shows the majority class and gets deeper as the box gets purer. `max_depth=3` here draws
only the top three levels of the depth-4 tree, and `impurity=False` hides the Gini numbers to save
space. The companion function `export_text` prints the same tree as text rules (used in Exercise 3).

Note that the numeric splits are on **z-scores**, because the pipeline scaled them. A split such as
`duration <= -0.78` means "duration at least 0.78 standard deviations below the mean". Exercise 3 shows
how to get rules in months instead.

In [ ]:
# Fetch the fitted depth-4 tree pipeline from section 9.
tree_pipeline = fitted["Decision tree (depth 4)"]
# Readable feature names: the encoded names with their step prefixes removed (a list comprehension).
tree_feature_names = [
    name.replace("numeric__", "").replace("categorical__", "")  # strip both prefixes
    for name in tree_pipeline.named_steps["preprocess"].get_feature_names_out()  # loop over all 44 names
]

# A very wide panel, because trees spread sideways.
fig, ax = plt.subplots(figsize=(17, 8))
# Draw the tree: each box shows the split rule, the sample count and the class mix.
plot_tree(
    tree_pipeline.named_steps["model"],  # the fitted DecisionTreeClassifier itself
    feature_names=tree_feature_names,  # label splits with readable names
    class_names=["Repaid", "Defaulted"],  # names for class 0 and class 1
    filled=True,  # colour boxes by majority class and purity
    rounded=True,  # rounded box corners
    fontsize=7,  # small text so the boxes fit
    max_depth=3,  # draw only the first three levels
    impurity=False,  # hide the Gini number in each box to reduce clutter
    ax=ax,  # draw into our panel
)
# Title.
ax.set_title("Decision tree, first three levels")
# Render the figure.
plt.show()

### Which features does the forest rely on?

Built-in tree importances are biased toward high-cardinality features.
**Permutation importance** is the more trustworthy measure: shuffle one
column in the test set and see how much the score falls.

#### Background: permutation importance

**Intuition.** If a feature matters, scrambling it should hurt the model. If the model never really
used it, scrambling it changes nothing.

**Algorithm.**

1. Score the fitted model on the test set: the baseline AUC.
2. For one column, randomly shuffle its values across rows. That breaks the link between the column
   and the outcome while keeping its distribution.
3. Re-score. The **importance** is baseline − shuffled score.
4. Repeat `n_repeats` times (20 here) and report the mean and standard deviation. Then do the same for
   every column.

**Why it beats built-in importance.** A forest's `feature_importances_` adds up impurity reductions on
the *training* data. That favours continuous and many-category features, which offer more split points,
and it rewards splits that only memorise. Permutation importance is measured on **held-out** data in
the units of a metric you care about (`scoring="roc_auc"`). It is also *model-agnostic*: it works for
any fitted pipeline.

**Detail worth noticing.** We pass the raw `X_test` to the *whole pipeline*, so each shuffle moves an
entire original column, such as all of `status`, rather than one dummy at a time. The importance is
reported per business variable.

**Caveat.** When two features are strongly correlated, shuffling one barely hurts because the model
falls back on the other, so both can look unimportant. **In finance** it is used to explain challenger
models to validation teams and to check that a model leans on sensible drivers.

Look for `status` far ahead of everything else (about 0.113 of AUC), then duration and amount.

In [ ]:
# permutation_importance measures how much a score drops when one column is shuffled.
from sklearn.inspection import permutation_importance

# The fitted random-forest pipeline.
forest = fitted["Random forest"]
# Shuffle each of the 18 raw columns 20 times on the TEST set and record the fall in ROC-AUC.
importance = permutation_importance(
    forest, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE,  # 20 shuffles per column for a stable average
    scoring="roc_auc",  # judge the damage by ROC-AUC rather than accuracy
)

# Collect the results into a sorted table.
importance_table = (
    pd.DataFrame(  # build the table from three equal-length arrays
        {  # column name -> values
            "feature": X_test.columns,  # raw column names (whole categorical columns, not dummies)
            "importance": importance.importances_mean,  # average AUC drop over 20 shuffles
            "std": importance.importances_std,  # how much the drop varied between shuffles
        }
    )
    .sort_values("importance", ascending=False)  # most important first
    .reset_index(drop=True)  # renumber rows 0, 1, 2, ...
)

# The top ten, to 4 decimal places.
print(importance_table.head(10).round(4).to_string(index=False))

The same importances as a bar chart. The thin grey error bars are ±1 standard deviation across the
20 shuffles. A bar whose error bar reaches zero, such as `savings` here, is not reliably different
from no importance at all.

In [ ]:
# The top 12 features, re-sorted ascending so the biggest bar ends up at the top.
plot_importance = importance_table.head(12).sort_values("importance")

# One panel.
fig, ax = plt.subplots(figsize=(9, 5.5))
# Bars = mean importance; xerr draws +/- one standard deviation as error bars.
ax.barh(plot_importance["feature"], plot_importance["importance"],
        xerr=plot_importance["std"], color="#2a78d6", height=0.65,  # error bars of +/- one std
        error_kw={"ecolor": "#52514e", "elinewidth": 1})  # grey, thin error bars
# Title says exactly what is measured.
ax.set_title("Permutation importance — drop in test ROC-AUC when the column is shuffled")
# Axis label.
ax.set_xlabel("Mean decrease in ROC-AUC")
# Vertical grid lines only.
ax.grid(axis="x")
# No horizontal grid lines.
ax.grid(axis="y", visible=False)
# Tidy spacing.
fig.tight_layout()
# Render the figure.
plt.show()

### Cross-validation

A single train/test split on 1,000 rows is noisy — change the random seed and
the AUC moves. Cross-validation fits the model $k$ times on different splits
and reports the spread, which is a far more honest summary.

#### Background: `StratifiedKFold` and `cross_val_score`

**Intuition.** Instead of one 750/250 split, rotate the test role around the data so that **every
applicant is in the test set exactly once**.

**Algorithm ($k$-fold cross-validation).**

1. Split the 1,000 rows into $k = 5$ folds of 200.
2. For each fold: fit on the other 4 folds (800 rows) and score on the held-out fold (200 rows).
3. Report the 5 scores, their **mean** (the performance estimate) and their **standard deviation**
   (how much the estimate depends on which rows you happened to test on).

**`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.**

- `n_splits=5` sets the number of folds.
- *Stratified* means each fold keeps the 30% default rate, as `stratify=y` did for the single split.
- `shuffle=True` randomises rows before cutting the folds, with `random_state` making that
  repeatable.

**`cross_val_score(pipeline, X, y, cv=folds, scoring="roc_auc")`** does the loop for you and returns an
array of 5 scores. Two important details:

- It **clones** the pipeline for each fold, making an unfitted copy, so the preprocessing is re-learned
  on each training fold. That is the `Pipeline` leakage guarantee at work.
- It is given the full `X, y`, not `X_train`, because it does its own splitting.

**In finance** this is standard for model selection and hyper-parameter tuning on cross-sectional data
such as applications or claims. For time-ordered data, use a walk-forward split instead (Notebook 05).

Look at the **± SD** next to each mean, and at the five individual fold scores.

In [ ]:
# StratifiedKFold makes the folds; cross_val_score fits and scores a model on each one.
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 5 folds, each keeping the 30% default rate; shuffle the rows first, with a fixed seed.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Heading.
print("5-fold cross-validated ROC-AUC (mean ± SD across folds)\n")
# Loop over the four fitted pipelines.
for name, pipeline in fitted.items():
    # On all 1,000 rows: train on 4 folds, score AUC on the 5th, five times. Returns 5 scores.
    scores = cross_val_score(pipeline, X, y, cv=folds, scoring="roc_auc")
    # "<26" left-aligns the name in 26 characters; mean ± standard deviation, then the five fold scores.
    print(f"  {name:<26} {scores.mean():.3f} ± {scores.std():.3f}"
          f"   folds: {np.round(scores, 3)}")  # adjacent f-strings join into one line

The fold-to-fold spread is often larger than the difference between models.
When that is true, claiming one model is better than another is not supported
by the data — and saying so out loud is a mark of competence, not weakness.

## 10. Choosing the cut-off from money, not from 0.5

`predict()` uses a 0.5 probability threshold. There is nothing special about
0.5 — it is a default, and it is almost always the wrong business choice,
because the two errors do not cost the same.

Set up the economics explicitly:

- **Approve, borrower repays**: the bank earns a margin on the loan.
- **Approve, borrower defaults**: the bank loses loss-given-default times the
  exposure.
- **Decline**: nothing gained, nothing lost.

Write $A$ for the loan amount, $M$ for the margin earned over the life of a
performing loan, $L$ for loss given default, and $p$ for the model's
predicted probability of default. Approving is worth taking if

$$(1-p)\,M A \;-\; p\,L A \;>\; 0 \qquad\Longleftrightarrow\qquad p \;<\; \frac{M}{M+L}$$

Two things fall out of that. First, the optimal cut-off has a closed form —
you do not have to search for it. Second, the loan amount $A$ **cancels**, so
under these assumptions the same threshold applies to every loan regardless
of size. (Add a fixed origination cost and it stops cancelling, which is
Exercise 4.)

We will compute the curve numerically anyway and check that it agrees.

#### Background: the expected-profit approval threshold $M/(M+L)$

**Intuition.** Approve a loan when, *on average*, it makes money. A bad loan costs much more than a
good loan earns: here 60% of the principal lost against 25% earned. So the bank should only approve
applicants whose default probability is comfortably below 50%.

**Derivation, one line at a time.** For one applicant with default probability $p$:

- Expected profit of approving: $(1-p)\,MA - p\,LA$.
- Expected profit of declining: $0$.
- Approve when $(1-p)MA - pLA > 0 \iff M - pM - pL > 0 \iff p < M/(M+L)$.

With $M = 0.25$ and $L = 0.60$, the cut-off is $0.25/0.85 \approx 0.294$. A **higher margin** raises the
cut-off (approve more) and a **higher loss** lowers it (approve less).

**The assumption that matters.** The formula treats `predict_proba` as a *true* probability, meaning
the model is **calibrated**: of the applicants scored 0.3, about 30% actually default. Logistic
regression is usually close, while forests and boosting often are not. If the probabilities are
biased, the empirically best cut-off drifts away from the formula.

**This cell** sets `MARGIN` and `LOSS_GIVEN_DEFAULT` and computes the formula cut-off. It then defines
`expected_profit`, which uses `np.where` to give each test loan its *realised* profit (−L × amount if
it actually defaulted, +M × amount if not) and sums over the approved loans. Finally it evaluates that
at 200 cut-offs from `np.linspace(0.02, 0.98, 200)` and picks the best with `argmax`. Compare the three
profit lines, and compare the grid's best cut-off with the formula's.

In [ ]:
# Assumed economics, as fractions of the loan amount.
MARGIN = 0.25              # interest earned over the life of a performing loan
LOSS_GIVEN_DEFAULT = 0.60  # share of principal lost when it does not

# The closed-form cut-off: approve if p < M / (M + L) = 0.25 / 0.85.
theoretical_threshold = MARGIN / (MARGIN + LOSS_GIVEN_DEFAULT)
# Print it to 3 decimals.
print(f"Closed-form optimal cut-off: {theoretical_threshold:.3f}")

# Loan amounts of the 250 test applicants as a NumPy array (the exposure A).
exposure = X_test["amount"].to_numpy()
# What actually happened to each test loan (1 = defaulted), as an array in the same order.
actually_defaulted = y_test.to_numpy()

# Realised profit of the test portfolio if we approve every applicant scored below `threshold`.
def expected_profit(probabilities, threshold):
    # Docstring describing the function.
    """Total profit if we approve every application scored below `threshold`."""
    # Boolean array: True where the predicted default probability is under the cut-off.
    approve = probabilities < threshold
    # np.where(condition, a, b): per loan, -L x amount if it defaulted, else +M x amount.
    profit = np.where(
        actually_defaulted == 1,  # condition: the loan went bad
        -LOSS_GIVEN_DEFAULT * exposure,  # value if True: the loss
        MARGIN * exposure,  # value if False: the margin earned
    )
    # Keep only the approved loans and add up their profit (declined loans contribute zero).
    return profit[approve].sum()


# 200 evenly spaced candidate cut-offs from 0.02 to 0.98.
thresholds = np.linspace(0.02, 0.98, 200)
# Portfolio profit at every candidate (a list comprehension turned into an array).
profit_curve = np.array([expected_profit(y_proba, t) for t in thresholds])

# argmax gives the position of the largest profit...
best_index = profit_curve.argmax()
# ...and this is the cut-off at that position.
best_threshold = thresholds[best_index]

# A cut-off above 1 approves everyone, since every probability is below 1.01.
approve_all = expected_profit(y_proba, 1.01)
# The profit implied by predict()'s 0.5 cut-off.
at_half = expected_profit(y_proba, 0.5)

# ">12,.0f" right-aligns in 12 characters with thousands separators and no decimals.
print(f"Approve every application:                 {approve_all:>12,.0f} DM")
# Profit at 0.5.
print(f"Model with the default 0.5 cut-off:        {at_half:>12,.0f} DM")
# Profit at the grid-search best cut-off.
print(f"Model at the best cut-off found ({best_threshold:.2f}):      {profit_curve[best_index]:>12,.0f} DM")
# How much the model plus a sensible cut-off adds over no screening at all.
print(f"\nImprovement over approving everyone:       {profit_curve[best_index] - approve_all:>12,.0f} DM")
# Compare the empirical and theoretical cut-offs.
print(f"\nGrid search found {best_threshold:.3f}; the formula says {theoretical_threshold:.3f}.")
# Spacing between neighbouring grid points: the finest difference the grid can resolve.
grid_step = thresholds[1] - thresholds[0]
# Gap between the two cut-offs, measured in grid steps.
print(f"The gap is {best_threshold - theoretical_threshold:.3f}, about "
      f"{(best_threshold - theoretical_threshold) / grid_step:.0f} grid steps of {grid_step:.3f}.")  # so not a rounding effect
# Profit if the bank simply uses the formula's cut-off.
at_formula = expected_profit(y_proba, theoretical_threshold)
# Compare it with the grid's best and with the 0.5 default.
print(f"Profit at the formula cut-off: {at_formula:,.0f} DM, which is "
      f"{profit_curve[best_index] - at_formula:,.0f} DM below the grid best "  # cost of using the formula
      f"and {at_formula - at_half:,.0f} DM above the 0.5 default.")  # gain over predict()'s cut-off

*Reading the last three lines.* The grid points are about 0.005 apart, yet the grid's best
cut-off (**0.353**) sits about 0.06, or roughly a dozen grid steps, above the formula (**0.294**).
So the two do not agree exactly, and the gap is not a rounding effect. It comes from two things.
First, there are only 250 test loans, so a handful of large loans near the cut-off can move the best
point. Second, the model's probabilities are not perfectly calibrated. The formula assumes they are.

The lesson survives, and the chart below shows why: the profit curve is **flat near its peak**. Using
the formula costs about 9,000 DM against the grid's best (56,486 vs 65,556 DM). It still earns about
20,600 DM more than the default 0.5 cut-off (35,885 DM), and far more than approving everyone
(−69,127 DM). The grid's best is also tuned on these very 250 loans, so it flatters itself, while the
formula used no test data at all. **The economics choose the cut-off region, not the model.** A
sample this small cannot pin down the third decimal place.

The next cell plots portfolio profit against the cut-off. The **green** line is the formula, the
**orange dotted** line is 0.5, the dashed grey line is approve-everyone, and the dot is the grid
maximum.

In [ ]:
# One wide panel.
fig, ax = plt.subplots(figsize=(9.5, 4.8))
# The profit curve, in thousands of DM.
ax.plot(thresholds, profit_curve / 1000, color="#2a78d6", linewidth=2)
# Dashed horizontal line: profit from approving everyone.
ax.axhline(approve_all / 1000, color="#52514e", linestyle="--", linewidth=1.2)
# Dotted vertical line at the default 0.5 cut-off.
ax.axvline(0.5, color="#eb6834", linestyle=":", linewidth=1.5)
# Solid green vertical line at the formula cut-off M/(M+L).
ax.axvline(theoretical_threshold, color="#1baf7a", linestyle="-", linewidth=1.2)
# A dot at the grid-search maximum; zorder=5 draws it on top of the lines.
ax.scatter([best_threshold], [profit_curve[best_index] / 1000], s=90, color="#eb6834",
           zorder=5, edgecolor="white", linewidth=1.5)  # white outline makes the dot stand out

# Label the dot, offset 10 points right and 14 down.
ax.annotate(f"best cut-off {best_threshold:.2f}",
            xy=(best_threshold, profit_curve[best_index] / 1000),  # the point being labelled
            xytext=(10, -14), textcoords="offset points", fontsize=9, color="#52514e")  # text offset in points
# Label the approve-everyone line near the left edge.
ax.annotate("approve everyone", xy=(0.03, approve_all / 1000), xytext=(0, 6),
            textcoords="offset points", fontsize=9, color="#52514e")  # offset measured in points, grey text
# Label the 0.5 line at the bottom of the plot (get_ylim()[0] is the lower y-limit).
ax.annotate("default 0.5", xy=(0.5, ax.get_ylim()[0]), xytext=(5, 12),
            textcoords="offset points", fontsize=9, color="#eb6834")  # orange to match the 0.5 line
# Label the formula line.
ax.annotate(f"M/(M+L) = {theoretical_threshold:.2f}",
            xy=(theoretical_threshold, ax.get_ylim()[0]), xytext=(-72, 26),  # text to the left of the line
            textcoords="offset points", fontsize=9, color="#1baf7a")  # green to match the formula line

# Title.
ax.set_title("Portfolio profit against the approval cut-off")
# x-axis: the decision rule.
ax.set_xlabel("Approve if predicted default probability is below this")
# y-axis: money.
ax.set_ylabel("Portfolio profit (thousand DM)")
# Render the figure.
plt.show()

Finally, the decision rule is turned into labels at the tuned cut-off:
`(y_proba >= best_threshold).astype(int)` marks anyone at or above the cut-off as a predicted default
(1). The cell then compares **recall on defaults** and the **number of approvals** with the 0.5 rule.
`matrix[1, 1] / matrix[1].sum()` is TP divided by all actual defaulters, which is recall. Look for
recall rising from **53.3% to 74.7%** while approvals fall from 189 to 156.

In [ ]:
# What does the confusion matrix look like at the chosen cut-off?
# Flag as "default" (1) every applicant at or above the tuned cut-off; astype(int) turns True/False into 1/0.
y_pred_tuned = (y_proba >= best_threshold).astype(int)
# The confusion matrix under the tuned rule.
tuned = confusion_matrix(y_test, y_pred_tuned)

# Recall on defaults = TP / (all actual defaults) = matrix[1, 1] / sum of row 1; approvals = count of p < 0.5.
print(f"At the 0.5 cut-off:                recall on defaults "
      f"{matrix[1, 1] / matrix[1].sum():.1%}, approvals {(y_proba < 0.5).sum()}")  # summing a True/False array counts the Trues
# The same two numbers at the tuned cut-off.
print(f"At the {best_threshold:.2f} cut-off:               recall on defaults "
      f"{tuned[1, 1] / tuned[1].sum():.1%}, approvals {(y_proba < best_threshold).sum()}")  # same two numbers, tuned rule
# The interpretation.
print("\nThe tuned model declines more applications and catches more bad loans.")
# The model is unchanged; only the decision rule moved.
print("Same model, same coefficients — only the cut-off changed.")

This is the single most practically useful idea in the notebook. Students
spend weeks tuning hyperparameters for a hundredth of an AUC point; moving the
threshold to match the actual cost of each error is usually worth far more.

It is also where **prescriptive** analytics begins, in Duran's taxonomy. The
model produces a probability; the business rule turns it into a decision.

### Exercise 2 (worked solution)

Rerun the profit analysis with `LOSS_GIVEN_DEFAULT = 0.35` — a secured loan
with collateral to recover. Predict which way the optimal cut-off moves
*before* you run it, using the formula, then confirm it numerically. Does the
direction match your intuition about secured versus unsecured lending?

**Approach.** Predict with the formula first: $M/(M+L)$ with a smaller $L$ has a smaller denominator, so
the cut-off must **rise**. Then repeat the section-10 grid search with $L = 0.35$. The constant
`LOSS_GIVEN_DEFAULT` is used again in Exercise 4, so the solution uses new names
(`LOSS_GIVEN_DEFAULT_SECURED`, `expected_profit_with_lgd`) and leaves it alone.

In [ ]:
# Exercise 2 — a secured loan: loss given default falls from 0.60 to 0.35.
# A new name, so the section-10 constant LOSS_GIVEN_DEFAULT (0.60) is untouched for later cells.
LOSS_GIVEN_DEFAULT_SECURED = 0.35

# Prediction BEFORE looking at data: the formula M/(M+L) with the smaller L.
threshold_secured = MARGIN / (MARGIN + LOSS_GIVEN_DEFAULT_SECURED)
# 0.25 / 0.85 versus 0.25 / 0.60.
print(f"Formula cut-off, unsecured (L = {LOSS_GIVEN_DEFAULT:.2f}): {theoretical_threshold:.3f}")
# A smaller loss per default lets the bank tolerate a higher default probability.
print(f"Formula cut-off, secured   (L = {LOSS_GIVEN_DEFAULT_SECURED:.2f}): {threshold_secured:.3f}  -> predicted to move UP")


# The section-10 profit function, with loss given default passed in as an argument instead of read from the constant.
def expected_profit_with_lgd(probabilities, threshold, lgd):
    # Docstring.
    """Realised test-portfolio profit approving every application scored below `threshold`."""
    # True for applications we approve.
    approve = probabilities < threshold
    # Per-loan profit: -lgd x amount for a default, +margin x amount for a good loan.
    profit = np.where(actually_defaulted == 1, -lgd * exposure, MARGIN * exposure)
    # Total over approved loans only.
    return profit[approve].sum()


# Profit at each of the same 200 grid cut-offs used in section 10.
profit_curve_secured = np.array(
    [expected_profit_with_lgd(y_proba, t, LOSS_GIVEN_DEFAULT_SECURED) for t in thresholds]  # one profit per cut-off
)
# Position of the best cut-off on the grid.
best_index_secured = profit_curve_secured.argmax()
# The best cut-off itself.
best_threshold_secured = thresholds[best_index_secured]

# Side-by-side summary: profit under each rule, for the secured loan.
print(f"\nSecured loan, approve everyone:        {expected_profit_with_lgd(y_proba, 1.01, LOSS_GIVEN_DEFAULT_SECURED):>10,.0f} DM")
# At the default 0.5 cut-off.
print(f"Secured loan, 0.5 cut-off:             {expected_profit_with_lgd(y_proba, 0.5, LOSS_GIVEN_DEFAULT_SECURED):>10,.0f} DM")
# At the formula cut-off.
print(f"Secured loan, formula cut-off ({threshold_secured:.3f}): {expected_profit_with_lgd(y_proba, threshold_secured, LOSS_GIVEN_DEFAULT_SECURED):>10,.0f} DM")
# At the grid-search best.
print(f"Secured loan, grid best ({best_threshold_secured:.3f}):       {profit_curve_secured[best_index_secured]:>10,.0f} DM")
# Compare the grid optimum with the unsecured one from section 10.
print(f"\nGrid-best cut-off: unsecured {best_threshold:.3f} -> secured {best_threshold_secured:.3f}")
# How many of the 250 applicants each formula cut-off approves.
print(f"Approvals at formula cut-off: unsecured {(y_proba < theoretical_threshold).sum()} -> secured {(y_proba < threshold_secured).sum()}")

# Plot both profit curves so the shift (and the flatness near the top) can be seen.
fig, ax = plt.subplots(figsize=(9.5, 4.5))
# Unsecured curve from section 10.
ax.plot(thresholds, profit_curve / 1000, color="#2a78d6", linewidth=2, label="Unsecured, L = 0.60")
# Secured curve.
ax.plot(thresholds, profit_curve_secured / 1000, color="#1baf7a", linewidth=2, label="Secured, L = 0.35")
# Formula cut-off, unsecured (dashed blue).
ax.axvline(theoretical_threshold, color="#2a78d6", linestyle="--", linewidth=1)
# Formula cut-off, secured (dashed green).
ax.axvline(threshold_secured, color="#1baf7a", linestyle="--", linewidth=1)
# Axis labels and title.
ax.set_xlabel("Approve if predicted default probability is below this")
# y-axis label.
ax.set_ylabel("Portfolio profit (thousand DM)")
# Title.
ax.set_title("Cheaper defaults move the profit-maximising cut-off to the right")
# Legend.
ax.legend(loc="lower right")
# Render the figure.
plt.show()

**Discussion.** The formula predicted the direction correctly before we ran anything. With $L = 0.35$
the cut-off rises from $0.25/0.85 = 0.294$ to $0.25/0.60 = 0.417$, and the formula cut-off approves
more applicants: **174** of the 250 instead of 142. When collateral limits the loss from a default, the bank can tolerate a riskier
borrower, so the approval cut-off moves up. That matches practice. Secured lending (mortgages, auto
loans) is extended to borrowers who would be declined for an unsecured personal loan.

The numerical check is more nuanced. The secured profit curve sits higher everywhere: approving
everyone now makes about **14,542 DM** instead of losing 69,127 DM, and the formula cut-off 0.417
earns about **76,944 DM**. But the grid-best cut-off barely moved, from **0.353 to 0.358**, where it
earns **77,801 DM**. The formula cut-off is only about 1% short of that. As in section 10, the curve is
flat near its peak and 250 test loans cannot resolve the exact optimum. The economics determine the
region, and the direction of the shift is unambiguous.

### Exercise 3 (worked solution)

The bank wants a scorecard it can put in front of a regulator. Fit a decision
tree with `max_depth=3`, print the rules with
`sklearn.tree.export_text(...)`, and write the top three rules out in plain
English as underwriting criteria. Then say what you gave up in AUC to get
that explainability.

**Approach.** Build a new pipeline with a depth-3 tree. For the scorecard, the numeric columns are
passed through **unscaled** (`"passthrough"` in the `ColumnTransformer`). Trees do not care about
scale, so the splits are the same, but the printed thresholds now read in months and DM rather than
z-scores. `export_text(..., show_weights=True)` prints the rules with `[repaid, defaulted]` training
counts at each leaf. The AUC is then compared with logistic regression on the test set and with 5-fold
cross-validation.

In [ ]:
# Exercise 3 — a depth-3 tree as a regulator-friendly scorecard.
# export_text prints a fitted tree as indented if/else rules.
from sklearn.tree import export_text

# Trees do not need scaling, so pass the numeric columns through unchanged ("passthrough"): split points then read in months and DM.
preprocessor_readable = ColumnTransformer(
    transformers=[  # list of (name, transformer, columns) triples
        ("numeric", "passthrough", numeric_columns),  # raw numbers, no z-scores
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_columns),  # same dummies as before
    ]
)
# Preprocessing plus a tree at most three questions deep.
scorecard_tree = Pipeline(
    steps=[  # list of (name, step) pairs, run in order
        ("preprocess", preprocessor_readable),  # step 1
        ("model", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),  # step 2: max_depth=3 caps the rules at 3 conditions
    ]
)
# Fit on the training rows only.
scorecard_tree.fit(X_train, y_train)

# Readable names for the encoded columns, prefixes stripped.
scorecard_names = [
    name.replace("numeric__", "").replace("categorical__", "")  # remove step prefixes
    for name in scorecard_tree.named_steps["preprocess"].get_feature_names_out()  # all 44 encoded columns
]
# Print the rules. show_weights=True shows [repaid, defaulted] training counts at each leaf.
print(export_text(scorecard_tree.named_steps["model"], feature_names=scorecard_names, show_weights=True))

# What did the explainability cost? Test-set ROC-AUC of the scorecard...
auc_scorecard = roc_auc_score(y_test, scorecard_tree.predict_proba(X_test)[:, 1])
# ...against the logistic regression on the same test set (y_proba from section 7).
auc_logistic = roc_auc_score(y_test, y_proba)
# And the more reliable 5-fold cross-validated AUCs, on the `folds` defined in the cross-validation section.
cv_scorecard = cross_val_score(scorecard_tree, X, y, cv=folds, scoring="roc_auc")
# Logistic regression on the same folds (clone gives an unfitted copy of the pipeline).
cv_logistic = cross_val_score(clone(logistic), X, y, cv=folds, scoring="roc_auc")

# Test-set comparison.
print(f"Test ROC-AUC:  depth-3 tree {auc_scorecard:.3f}   logistic regression {auc_logistic:.3f}   cost {auc_logistic - auc_scorecard:.3f}")
# Cross-validated comparison.
print(f"5-fold CV AUC: depth-3 tree {cv_scorecard.mean():.3f}   logistic regression {cv_logistic.mean():.3f}   cost {cv_logistic.mean() - cv_scorecard.mean():.3f}")

**Discussion.** The tree's first split is checking-account status, the dominant variable in every
analysis in this notebook. Three underwriting rules read off its leaves, with training counts as
`[repaid, defaulted]`:

1. **No checking account at this bank, no other instalment plans, no past payment delays:
   low risk, approve.** 208 repaid and 12 defaulted, a default rate of about 5%, the safest group in
   the tree.
2. **Has a checking account and the loan is 11 months or shorter, unless it is for
   retraining: approve.** 60 repaid and 11 defaulted, about 15%. Short loans to account holders are
   low risk.
3. **Has a checking account and the loan runs 12 months or more, with known savings: refer
   for manual review.** 166 repaid and 161 defaulted, roughly 50/50. That is far above the 30%
   portfolio rate, and it is the riskiest large segment in the book. (The small
   retraining-purpose leaf, 1 repaid and 5 defaulted, is the only one the tree labels "Defaulted", but
   it is too small to be a policy.)

**The cost of explainability.** The depth-3 tree reached a test ROC-AUC of about **0.691** against
**0.800** for logistic regression, losing about **0.11** of AUC. With 5-fold cross-validation the
figures are about **0.701** against **0.778**, a loss of roughly 0.08. That is a large price,
considerably more than the fold-to-fold noise. Logistic regression is also explainable, through its
odds ratios, and it does not pay this price. That is the honest reason scorecards are built on it
rather than on shallow trees.

### Exercise 4 (worked solution)

Add a fixed origination cost of 150 DM to every approved loan, regardless of
size. Re-derive the approval rule on paper: the loan amount no longer
cancels, so the cut-off now depends on $A$. Implement it — approve when
$(1-p)MA - pLA - 150 > 0$ — and compare total portfolio profit against the
single-threshold rule. Which loans does the new rule decline that the old one
approved?

**Derivation.** With a fixed cost $C$ per approved loan, approve when

$$(1-p)MA - pLA - C > 0 \iff p < \frac{MA - C}{(M+L)A} = \frac{M}{M+L} - \frac{C}{(M+L)A}.$$

The cut-off is the old $M/(M+L)$ *minus* a penalty that shrinks as the loan amount $A$ grows. Large
loans face nearly the old cut-off, and small loans face a much stricter one. Below $A = C/M$ the
cut-off is negative, so the loan is never worth making. The solution applies this per-loan cut-off
and charges both rules the 150 DM fee so that the comparison is fair.

In [ ]:
# Exercise 4 — a fixed origination cost of 150 DM per approved loan.
# A new constant; nothing earlier is changed.
ORIGINATION_COST = 150

# Expected profit of approving a loan of size A with default probability p is (1-p)MA - pLA - C.
# Setting it > 0 and solving for p gives p < (MA - C) / ((M + L) A): a cut-off that depends on the amount A.
amount_cutoff = (MARGIN * exposure - ORIGINATION_COST) / ((MARGIN + LOSS_GIVEN_DEFAULT) * exposure)
# Below A = C/M the cut-off is negative, so such a loan is never worth making.
print(f"Break-even loan size C/M: {ORIGINATION_COST / MARGIN:,.0f} DM (smaller loans can never cover the fee)")

# Old rule: one cut-off M/(M+L) for every loan, ignoring the fee.
approve_old = y_proba < theoretical_threshold
# New rule: each loan gets its own cut-off, which rises towards M/(M+L) as the amount grows.
approve_new = y_proba < amount_cutoff

# Realised profit per test loan, now net of the 150 DM fee that every approved loan incurs.
profit_net = np.where(
    actually_defaulted == 1,  # loan went bad
    -LOSS_GIVEN_DEFAULT * exposure,  # lose L x amount
    MARGIN * exposure,  # otherwise earn M x amount
) - ORIGINATION_COST  # either way, the fee is paid on approval

# Compare the two rules on the same 250 test applicants, both charged the fee.
print(f"\nSingle cut-off {theoretical_threshold:.3f}:  {approve_old.sum():>3} approved, profit {profit_net[approve_old].sum():>10,.0f} DM")
# The amount-dependent rule.
print(f"Amount-dependent cut-off: {approve_new.sum():>3} approved, profit {profit_net[approve_new].sum():>10,.0f} DM")
# Difference between the two.
print(f"Gain from the new rule:                  {profit_net[approve_new].sum() - profit_net[approve_old].sum():>+10,.0f} DM")

# Loans the old rule approved but the new rule declines (& = "and", ~ = "not").
switched = approve_old & ~approve_new
# A small table describing those loans, using new names only.
switched_loans = pd.DataFrame(
    {  # column name -> array of values for the switched loans
        "amount": exposure[switched],  # loan size in DM
        "p_default": y_proba[switched].round(3),  # model's default probability
        "own_cutoff": amount_cutoff[switched].round(3),  # the amount-dependent cut-off for that loan
        "defaulted": actually_defaulted[switched],  # what actually happened
        "net_profit": profit_net[switched].round(0),  # what approving it actually earned, after the fee
    }
).sort_values("amount")  # smallest loans first

# Summary of the switched loans.
print(f"\nLoans approved by the old rule but declined by the new one: {switched.sum()}")
# Their size range and median against the median approved loan under the old rule.
print(f"Their amounts: {switched_loans['amount'].min():,} to {switched_loans['amount'].max():,} DM, "
      f"median {switched_loans['amount'].median():,.0f} DM (median old-rule approval: {np.median(exposure[approve_old]):,.0f} DM)")  # second half of the message
# How many of them actually defaulted, and what they earned in total.
print(f"Of these, {switched_loans['defaulted'].sum()} defaulted; together they earned {switched_loans['net_profit'].sum():,.0f} DM net of fees")
# Show the ten smallest.
print("\nThe ten smallest:")
# to_string(index=False) hides the row numbers.
print(switched_loans.head(10).to_string(index=False))

**Discussion.** Solving $(1-p)MA - pLA - 150 > 0$ gives a cut-off of $p < 0.294 - 150/(0.85A)$. On the
250 test loans, the amount-dependent rule approves **109** loans against **142** for the single 0.294
cut-off, and earns about **36,514 DM** against **35,186 DM** net of fees, a gain of about **1,329 DM**.

The new rule declines **33** loans that the old rule approved, and all of them are **small**. They
range from 250 to 3,595 DM, with a median of about 1,103 DM, against a median old-rule approval of
about 1,924 DM. A fixed fee is a large fraction of a small loan, so a small loan needs a much lower
default probability to cover it. Any loan below $C/M = 600$ DM can never cover the fee. Of the 33,
**6** actually defaulted, and together the 33 lost about 1,329 DM net of fees. Declining them is
exactly where the gain comes from.

A caveat in the same spirit as Exercise 2. Both rules here use the *formula* cut-off. The grid-best
single cut-off from section 10 (0.35) would have earned more on this particular test set, again
because the model is not perfectly calibrated and 250 loans is a small sample. The economic lesson
stands: **with fixed costs, loan size belongs in the approval rule**. That is why real lenders set
minimum loan amounts and price small loans differently.

---

## Recap

1. Explore before you model; category-level default rates carry most of the
   story.
2. Some predictive variables are unlawful to use. Dropping them is necessary
   and not sufficient.
3. Put preprocessing inside a `Pipeline` so leakage becomes impossible rather
   than merely discouraged.
4. Accuracy is close to useless on imbalanced data. Use ROC-AUC, precision,
   recall, and look at the confusion matrix.
5. Logistic regression is still the default in regulated lending because a
   coefficient is an explanation.
6. A training-test accuracy gap is overfitting.
7. Cross-validate; if the fold spread exceeds the gap between models, you do
   not have a winner.
8. Set the cut-off from expected profit, not from 0.5. Under simple
   assumptions it is just $M/(M+L)$.

---

## Your turn

The code in this notebook is complete, including worked solutions to all four exercises. Your job:

1. **Run every cell** from top to bottom (*Run All*, or Shift+Enter through the notebook) and check
   that your output matches the numbers quoted in the text.
2. **Read** the explanations, the background sections and the worked solutions with their
   discussions. Make sure you can say what each line of code does and why.
3. **Take the Canvas comprehension quiz** for Notebook 04. It asks about the code, the methods
   (logistic regression, trees, forests, boosting, cross-validation, ROC-AUC, the profit cut-off) and
   the finance lessons.
4. **Save and push your run notebook**, with its outputs, to your repository:

```
git add notebooks/04_ml_credit_risk.ipynb
git commit -m "Lab 3: ran notebook 04"
git push
```

Next: `05_ml_market_prediction.ipynb` — the same machinery, applied to a
problem where the signal is much weaker, and what to do about that.